In [2]:
from sentence_transformers import SentenceTransformer

# Модель скачается автоматически при инициализации
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Проверка работы
embeddings = model.encode(["Привет, мир!", "Hello world!"])
print(embeddings.shape)

c:\Users\Gehopp\Desktop\forgit\RT_second\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1576.06it/s]


(2, 384)


In [14]:
#!/usr/bin/env python3
"""
Извлечение чанков (функций/классов) из всех .py файлов, лежащих в поддиректориях
относительно текущей рабочей директории. Файлы, находящиеся прямо в текущей директории,
игнорируются (не учитываются). Результат сохраняется в JSON-файл в текущей директории.
Формат chunk_id: {relative_path}:{name}:{start_line}
Устойчив к ошибкам кодировки, синтаксическим ошибкам, бинарным файлам.
"""

import ast
import json
import logging
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)


class ChunkExtractor:
    """Извлекает чанки из Python-файлов, игнорируя файлы в корневой директории"""

    def __init__(self, root_dir: Path):
        """
        Args:
            root_dir: Корневая директория (текущая рабочая директория).
                      Файлы из этой директории игнорируются, обрабатываются только поддиректории.
        """
        self.root_dir = root_dir.resolve()
        self.chunks: List[Dict[str, Any]] = []

    def _should_process_file(self, file_path: Path) -> bool:
        """
        Определяет, нужно ли обрабатывать файл.
        Возвращает True, если файл находится в поддиректории (не прямо в root_dir).
        """
        try:
            rel = file_path.relative_to(self.root_dir)
            # Если относительный путь не содержит разделителя (т.е. файл прямо в корне) – пропускаем
            return rel.as_posix().find('/') != -1
        except ValueError:
            # Файл вне root_dir (маловероятно) – тоже пропускаем
            return False

    def _get_relative_path(self, file_path: Path) -> str:
        """Возвращает относительный путь от root_dir, с прямыми слэшами"""
        rel = file_path.relative_to(self.root_dir)
        return rel.as_posix()

    def _get_node_source(self, source_code: str, node: ast.AST) -> str:
        """Извлекает исходный код узла, безопасно обрабатывая отсутствие end_lineno"""
        lines = source_code.splitlines()
        start = node.lineno - 1
        if hasattr(node, 'end_lineno') and node.end_lineno is not None:
            end = node.end_lineno
        else:
            end = len(lines)
        end = min(end, len(lines))
        if start >= end:
            return ""
        return "\n".join(lines[start:end])

    def _extract_chunks_from_node(self, node: ast.AST, source_code: str, rel_path: str,
                                  parent_class_name: Optional[str] = None) -> List[Dict]:
        """Рекурсивно извлекает чанки из AST-узла"""
        chunks = []
        if isinstance(node, ast.ClassDef):
            name = node.name
            if parent_class_name:
                name = f"{parent_class_name}.{name}"
            chunk_id = f"{rel_path}:{name}:{node.lineno}"
            source = self._get_node_source(source_code, node)
            chunks.append({
                "chunk_id": chunk_id,
                "source_code": source,
                "type": "class",
                "name": name,
                "start_line": node.lineno,
                "end_line": getattr(node, 'end_lineno', node.lineno),
                "docstring": ast.get_docstring(node)
            })
            for child in ast.iter_child_nodes(node):
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    method_name = f"{node.name}.{child.name}"
                    method_chunks = self._extract_chunks_from_node(
                        child, source_code, rel_path, parent_class_name=node.name
                    )
                    for ch in method_chunks:
                        ch["chunk_id"] = f"{rel_path}:{method_name}:{child.lineno}"
                        ch["name"] = method_name
                        ch["type"] = "method"
                    chunks.extend(method_chunks)
                elif isinstance(child, ast.ClassDef):
                    nested_chunks = self._extract_chunks_from_node(
                        child, source_code, rel_path, parent_class_name=node.name
                    )
                    chunks.extend(nested_chunks)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and parent_class_name is None:
            name = node.name
            chunk_id = f"{rel_path}:{name}:{node.lineno}"
            source = self._get_node_source(source_code, node)
            chunks.append({
                "chunk_id": chunk_id,
                "source_code": source,
                "type": "function",
                "name": name,
                "start_line": node.lineno,
                "end_line": getattr(node, 'end_lineno', node.lineno),
                "docstring": ast.get_docstring(node)
            })
        return chunks

    def process_file(self, file_path: Path) -> List[Dict]:
        """Обрабатывает один .py файл, возвращает список чанков"""
        if not self._should_process_file(file_path):
            logger.debug(f"Ignoring file in root directory: {file_path}")
            return []

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                source_code = f.read()
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='latin-1') as f:
                    source_code = f.read()
                logger.warning(f"File {file_path} decoded with latin-1")
            except Exception as e:
                logger.error(f"Cannot read {file_path}: {e}")
                return []
        except Exception as e:
            logger.error(f"Cannot read {file_path}: {e}")
            return []

        try:
            tree = ast.parse(source_code, filename=str(file_path))
        except SyntaxError as e:
            logger.warning(f"Syntax error in {file_path}: {e}. Skipping.")
            return []
        except Exception as e:
            logger.error(f"AST parse error in {file_path}: {e}")
            return []

        rel_path = self._get_relative_path(file_path)
        chunks = []
        for node in ast.iter_child_nodes(tree):
            node_chunks = self._extract_chunks_from_node(node, source_code, rel_path, None)
            chunks.extend(node_chunks)
        logger.debug(f"Processed {file_path}: {len(chunks)} chunks")
        return chunks

    def process_all(self) -> List[Dict]:
        """Обходит все поддиректории рекурсивно, собирает чанки"""
        # Находим все .py файлы, включая глубокие подпапки
        py_files = list(self.root_dir.rglob("*.py"))
        logger.info(f"Found {len(py_files)} .py files in total")

        all_chunks = []
        for py_file in py_files:
            chunks = self.process_file(py_file)
            all_chunks.extend(chunks)

        logger.info(f"Total chunks extracted: {len(all_chunks)}")
        self.chunks = all_chunks
        return all_chunks

    def save_chunks(self, output_path: Path):
        """Сохраняет чанки в JSON-файл"""
        data = {
            "root_dir": str(self.root_dir),
            "num_chunks": len(self.chunks),
            "chunks": self.chunks
        }
        try:
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            logger.info(f"Chunks saved to {output_path}")
        except Exception as e:
            logger.error(f"Failed to save: {e}")


def main():
    # Текущая рабочая директория – корень, но файлы в ней игнорируются
    current_dir = Path.cwd()
    logger.info(f"Working directory (root): {current_dir}")
    logger.info("Only .py files inside subdirectories will be processed (files in root are ignored).")

    extractor = ChunkExtractor(root_dir=current_dir)
    chunks = extractor.process_all()

    output_file = current_dir / "extracted_chunks.json"
    extractor.save_chunks(output_file)

    if not chunks:
        print("\nВнимание: не найдено ни одного чанка в поддиректориях.")
        print("Убедитесь, что в текущей директории есть подпапки с Python-кодом.")
    else:
        print(f"\nИзвлечено {len(chunks)} чанков. Примеры (первые 5):")
        for chunk in chunks[:5]:
            print(f"  {chunk['chunk_id']}  (тип: {chunk['type']})")
        print(f"\nРезультат сохранён в {output_file}")


if __name__ == "__main__":
    main()

2026-06-05 23:40:03,175 - INFO - Working directory (root): c:\Users\Gehopp\Desktop\forgit\RT_second
2026-06-05 23:40:03,177 - INFO - Only .py files inside subdirectories will be processed (files in root are ignored).
2026-06-05 23:40:04,662 - INFO - Found 12202 .py files in total
2026-06-05 23:41:50,972 - WARNING - File C:\Users\Gehopp\Desktop\forgit\RT_second\.venv\Lib\site-packages\joblib\test\test_func_inspect_special_encoding.py decoded with latin-1
2026-06-05 23:41:56,787 - INFO - Total chunks extracted: 99991
2026-06-05 23:42:00,939 - INFO - Chunks saved to c:\Users\Gehopp\Desktop\forgit\RT_second\extracted_chunks.json



Извлечено 99991 чанков. Примеры (первые 5):
  extracted_gymhero/gymhero/gymhero/config.py:Settings:11  (тип: class)
  extracted_gymhero/gymhero/gymhero/config.py:ContainerDevSettings:37  (тип: class)
  extracted_gymhero/gymhero/gymhero/config.py:ContainerTestSettings:44  (тип: class)
  extracted_gymhero/gymhero/gymhero/config.py:LocalTestSettings:51  (тип: class)
  extracted_gymhero/gymhero/gymhero/config.py:LocalDevSettings:58  (тип: class)

Результат сохранён в c:\Users\Gehopp\Desktop\forgit\RT_second\extracted_chunks.json


In [17]:
#!/usr/bin/env python3
"""
Скрипт для генерации эмбеддингов чанков кода с помощью sentence-transformers
и сохранения их в ChromaDB.
"""

import json
import logging
from pathlib import Path
from typing import List, Dict, Any

import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.utils import embedding_functions

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Конфигурация
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHUNKS_JSON_PATH = Path("./extracted_chunks.json")      # путь к файлу с чанками
CHROMA_PERSIST_DIR = Path("./chroma_db")                # куда сохранять базу
COLLECTION_NAME = "code_chunks"


def get_text_for_embedding(chunk: Dict[str, Any]) -> str:
    """
    Формирует текст для генерации эмбеддинга чанка.
    Комбинирует: тип, имя, docstring (если есть) и начало исходного кода.
    """
    chunk_type = chunk.get("type", "unknown")
    name = chunk.get("name", "")
    docstring = chunk.get("docstring") or ""
    source_code = chunk.get("source_code", "")

    # Ограничиваем длину кода (первые 1000 символов достаточно для семантики)
    code_snippet = source_code[:1000]

    parts = [
        f"{chunk_type}: {name}",
        f"docstring: {docstring}" if docstring else "",
        f"code: {code_snippet}"
    ]
    return "\n".join(part for part in parts if part).strip()


def load_chunks(json_path: Path) -> List[Dict[str, Any]]:
    """Загружает список чанков из JSON-файла."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # Поддерживаем оба возможных формата: прямой список или поле "chunks"
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and "chunks" in data:
        return data["chunks"]
    else:
        raise ValueError("Неожиданный формат JSON. Ожидается список или объект с ключом 'chunks'.")


def main():
    # 1. Проверка существования файла с чанками
    if not CHUNKS_JSON_PATH.exists():
        logger.error(f"Файл {CHUNKS_JSON_PATH} не найден. Сначала запустите извлечение чанков.")
        return

    # 2. Загрузка чанков
    chunks = load_chunks(CHUNKS_JSON_PATH)
    logger.info(f"Загружено {len(chunks)} чанков")

    if not chunks:
        logger.warning("Нет чанков для индексации.")
        return

    # 3. Инициализация модели sentence-transformers
    logger.info(f"Загрузка модели {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)
    logger.info("Модель загружена")

    # 4. Генерация эмбеддингов
    texts = [get_text_for_embedding(chunk) for chunk in chunks]
    logger.info("Генерация эмбеддингов...")
    embeddings = model.encode(texts, show_progress_bar=True)
    logger.info(f"Эмбеддинги сгенерированы, форма: {embeddings.shape}")

    # 5. Подготовка метаданных и ID для ChromaDB
    ids = []
    metadatas = []
    for i, chunk in enumerate(chunks):
        # ID должен быть уникальным и состоять только из ASCII-символов (рекомендация ChromaDB)
        chunk_id = chunk.get("chunk_id", f"chunk_{i}")
        # Заменяем недопустимые символы в ID (хотя ChromaDB позволяет многое, но лучше безопасно)
        safe_id = chunk_id.replace(":", "_").replace("/", "_").replace(".", "_")
        ids.append(safe_id)

        metadatas.append({
            "chunk_id": chunk_id,          # исходный идентификатор для оценки
            "type": chunk.get("type", ""),
            "name": chunk.get("name", ""),
            "file_path": chunk.get("file_path", ""),
            "start_line": chunk.get("start_line", 0),
            "end_line": chunk.get("end_line", 0),
        })

    # 6. Инициализация ChromaDB (постоянное хранилище)
    CHROMA_PERSIST_DIR.mkdir(exist_ok=True)
    client = chromadb.PersistentClient(path=str(CHROMA_PERSIST_DIR))

    # Удаляем коллекцию, если она уже существует (для переиндексации)
    try:
        client.delete_collection(COLLECTION_NAME)
        logger.info(f"Старая коллекция '{COLLECTION_NAME}' удалена")
    except Exception:
        pass

    # Создаём новую коллекцию с функцией эмбеддингов по умолчанию (мы уже передаём готовые векторы)
    # Но проще использовать встроенный эмбеддер, если хотим автоматическое кэширование.
    # Для полного контроля передадим готовые векторы через .add(embeddings=...)
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}   # используем косинусное расстояние
    )

    # Добавляем документы в коллекцию
    # ChromaDB принимает список эмбеддингов в виде списка списков
    logger.info("Добавление в ChromaDB...")
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),   # конвертируем numpy в list
        metadatas=metadatas,
        documents=texts,                  # сохраняем исходный текст для отладки
    )
    logger.info(f"Успешно добавлено {len(ids)} векторов в коллекцию '{COLLECTION_NAME}'")

    # 7. Небольшой тест поиска (опционально)
    test_query = "как создать токен доступа"
    test_embedding = model.encode([test_query])
    results = collection.query(
        query_embeddings=test_embedding.tolist(),
        n_results=3
    )
    logger.info("Пример поиска:")
    for i, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
        print(f"  {i+1}. {meta['chunk_id']} (distance={dist:.4f})")


if __name__ == "__main__":
    main()

2026-06-05 23:50:55,564 - INFO - Загружено 99991 чанков
2026-06-05 23:50:55,569 - INFO - Загрузка модели sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...
2026-06-05 23:50:55,579 - INFO - No device provided, using cpu
2026-06-05 23:50:56,100 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-05 23:50:56,159 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
2026-06-05 23:50:56,379 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-05 23:50:56,427 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/para

InternalError: ValueError: Batch size of 99991 is greater than max batch size of 5461

In [ ]:
#!/usr/bin/env python3
"""
Оценка Precision@5 с помощью скрипта score.py.
Выполняет поиск по вопросам из eval_questions.json,
сохраняет результаты в results.json и запускает score.py.
"""

import json
import subprocess
import sys
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

# Конфигурация
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHROMA_PATH = Path("./chroma_db")
COLLECTION_NAME = "code_chunks"
QUESTIONS_PATH = Path("./eval_questions.json")
RESULTS_PATH = Path("./results.json")
SCORE_SCRIPT = Path("./score.py")          # путь к скрипту скорера из датасета
TOP_K = 5


def load_questions() -> list:
    """Загружает вопросы из eval_questions.json."""
    with open(QUESTIONS_PATH, 'r', encoding='utf-8') as f:
        return json.load(f)


def get_chunk_id_from_metadata(metadata: dict) -> str:
    """Извлекает исходный chunk_id из метаданных ChromaDB."""
    return metadata.get("chunk_id", "")


def search(query: str, collection, model, top_k: int = TOP_K) -> list:
    """
    Выполняет семантический поиск по запросу.
    Возвращает список chunk_id (строки) в порядке релевантности.
    """
    # Получаем эмбеддинг запроса
    query_embedding = model.encode([query])
    # Поиск в ChromaDB
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    # Извлекаем chunk_id из метаданных
    chunk_ids = []
    metadatas = results['metadatas'][0] if results['metadatas'] else []
    for meta in metadatas:
        chunk_id = get_chunk_id_from_metadata(meta)
        chunk_ids.append(chunk_id)
    return chunk_ids


def generate_results(questions: list, collection, model) -> list:
    """Формирует список результатов для каждого вопроса."""
    results = []
    for q in questions:
        qid = q["question_id"]
        query_text = q["query"]
        top5 = search(query_text, collection, model, TOP_K)
        results.append({
            "question_id": qid,
            "top_5_chunks": top5
        })
        print(f"Processed {qid}: {query_text[:50]}... -> found {len(top5)} chunks")
    return results


def save_results(results: list):
    """Сохраняет results.json в нужном формате."""
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(results)} results to {RESULTS_PATH}")


def run_score_script() -> float:
    """Запускает score.py и возвращает значение Precision@5."""
    if not SCORE_SCRIPT.exists():
        print(f"Ошибка: {SCORE_SCRIPT} не найден. Укажите правильный путь.")
        return 0.0
    try:
        result = subprocess.run(
            [sys.executable, str(SCORE_SCRIPT), "--predictions", str(RESULTS_PATH), "--questions", str(QUESTIONS_PATH)],
            capture_output=True,
            text=True,
            check=False
        )
        print(result.stdout)
        if result.stderr:
            print("Stderr:", result.stderr)
        # Парсим строку "Total score: 0.xxx" из вывода
        for line in result.stdout.splitlines():
            if "Total score:" in line:
                parts = line.split()
                try:
                    score = float(parts[-1])
                    return score
                except ValueError:
                    pass
        return 0.0
    except Exception as e:
        print(f"Ошибка при запуске score.py: {e}")
        return 0.0


def main():
    # 1. Подключение к ChromaDB
    if not CHROMA_PATH.exists():
        print(f"База ChromaDB не найдена по пути {CHROMA_PATH}. Сначала запустите индексацию.")
        sys.exit(1)

    client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"Коллекция '{COLLECTION_NAME}' загружена, количество документов: {collection.count()}")
    except Exception as e:
        print(f"Не удалось получить коллекцию: {e}")
        sys.exit(1)

    # 2. Загрузка модели
    print(f"Загрузка модели {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)

    # 3. Загрузка вопросов
    if not QUESTIONS_PATH.exists():
        print(f"Файл {QUESTIONS_PATH} не найден.")
        sys.exit(1)
    questions = load_questions()
    print(f"Загружено {len(questions)} вопросов.")

    # 4. Генерация предсказаний
    results = generate_results(questions, collection, model)

    # 5. Сохранение results.json
    save_results(results)

    # 6. Запуск скорера и вывод метрики
    precision = run_score_script()
    print(f"\nИтоговая Precision@5: {precision:.3f}")


if __name__ == "__main__":
    main()